In [17]:
import pandas as pd
import requests
import re

# DWD-Datei laden
url = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt"
response = requests.get(url)
response.encoding = "latin1"
lines = response.text.splitlines()

# Ab Zeile 3 (nach Header + Bindestrichen)
data_lines = lines[2:]

# Bekannte Bundesländer (DWD-Standard)
bundeslaender = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen",
    "Hamburg", "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen",
    "Nordrhein-Westfalen", "Rheinland-Pfalz", "Saarland",
    "Sachsen", "Sachsen-Anhalt", "Schleswig-Holstein", "Thüringen"
]

# DataFrame-Spalten
columns = [
    "Stations_id",
    "von_datum",
    "bis_datum",
    "Stationshoehe",
    "geoBreite",
    "geoLaenge",
    "Stationsname",
    "Bundesland"
]

rows = []
for line in data_lines:
    if not line.strip():
        continue
    parts = line.split(maxsplit=6)
    if len(parts) < 7:
        continue

    last_field = parts[6]

    # Bundesland extrahieren (sofern vorhanden)
    bundesland = None
    for bl in bundeslaender:
        if re.search(rf"\b{re.escape(bl)}\b", last_field):
            bundesland = bl
            break

    # Stationsname = alles vor dem Bundesland
    if bundesland:
        station_name = last_field.split(bundesland)[0].strip()
    else:
        station_name = last_field.strip()

    # Entferne optionales "Frei."
    station_name = re.sub(r"\bFrei\.?$", "", station_name).strip()

    # Zeile hinzufügen
    rows.append(parts[:6] + [station_name, bundesland])

# DataFrame erstellen
df = pd.DataFrame(rows, columns=columns)

# Zahlen konvertieren, wo sinnvoll
numeric_cols = ["Stations_id", "Stationshoehe", "geoBreite", "geoLaenge"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df.head(10))
print("\nSpalten:", df.columns.tolist())


,Stations_id,von_datum,bis_datum,Stationshoehe,geoBreite,geoLaenge,Stationsname,Bundesland
0,1,19370101,19860630,478,47.8413,8.8493,Aach,Baden-Württemberg
1,3,18910101,20110331,202,50.7827,6.0941,Aachen,Nordrhein-Westfalen
2,11,19800901,20251020,680,47.9736,8.5205,Donaueschingen (Landeplatz),Baden-Württemberg
3,44,19690101,20251020,44,52.9336,8.2370,Großenkneten,Niedersachsen
4,52,19690101,20011231,46,53.6623,10.1990,Ahrensburg-Wulfsdorf,Schleswig-Holstein
5,61,19750701,19780831,339,48.8443,12.6171,Aiterhofen,Bayern
6,70,19730601,19860930,712,48.2052,9.0371,Albstadt-Ebingen,Baden-Württemberg
7,71,19861101,20191231,759,48.2156,8.9784,Albstadt-Badkap,Baden-Württemberg
8,72,19780901,19950531,794,48.2766,9.0001,Albstadt-Onstmettingen,Baden-Württemberg
9,73,19590301,20251020,374,48.6183,13.0620,Aldersbach-Kramersepp,Bayern



Spalten: ['Stations_id', 'von_datum', 'bis_datum', 'Stationshoehe', 'geoBreite', 'geoLaenge', 'Stationsname', 'Bundesland']


In [ ]:
# ================================================================
# Urban Heat Island Analyse – Hitzetage & Tropennächte 2019–2024 (DWD Historical)
# ================================================================

import pandas as pd
import numpy as np
import requests, zipfile, io, re, os
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from tqdm import tqdm

# ------------------------------------------------
# 1️ Städtedaten laden
# ------------------------------------------------
excel_path = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\05-staedte.xlsx"

print(" Lade Städtedaten ...")
cities = pd.read_excel(excel_path, sheet_name="Städte", skiprows=1)
cities = cities.rename(columns={
    "Stadt": "city",
    "Fläche km² ¹⁾": "area_km2",
    "Bevölkerung auf Grundlage des ZENSUS 2022 ²⁾ insgesamt": "population"
})
cities = cities[["city", "area_km2", "population"]].dropna()

# N größte Städte
N = 300
cities = cities.nlargest(N, "population").reset_index(drop=True)
print(f"\n Verwende {N} größte Städte:")
print(cities)

# ------------------------------------------------
# 2️ Geokoordinaten holen
# ------------------------------------------------
print("\n Ermittle Geokoordinaten ...")
geolocator = Nominatim(user_agent="uhi_analysis_germany")

def get_coords(city):
    try:
        clean = re.sub(r",.*", "", city)
        loc = geolocator.geocode(f"{clean}, Germany", timeout=10)
        if loc:
            return pd.Series([loc.latitude, loc.longitude, clean])
    except:
        pass
    return pd.Series([np.nan, np.nan, city])

tqdm.pandas()
cities[["lat", "lon", "city_clean"]] = cities["city"].progress_apply(get_coords)

# ------------------------------------------------
# 3️ Lade DWD-Stationen (historisch)
# ------------------------------------------------
print("\n Lade DWD-Stationsbeschreibung (historische Tageswerte)...")

url_st = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt"

response = requests.get(url_st)
response.encoding = "latin1"
lines = response.text.splitlines()

# Bekannte Bundesländer
bundeslaender = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen",
    "Hamburg", "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen",
    "Nordrhein-Westfalen", "Rheinland-Pfalz", "Saarland",
    "Sachsen-Anhalt", "Sachsen", "Schleswig-Holstein", "Thüringen"
]

stations_rows = []
for line in lines[2:]:  # ab Zeile 3
    if not line.strip():
        continue
    parts = line.split(maxsplit=6)
    if len(parts) < 7:
        continue

    rest = parts[6]
    bundesland = next((bl for bl in bundeslaender if re.search(rf"\b{re.escape(bl)}\b", rest)), None)
    station_name = rest.split(bundesland)[0].strip() if bundesland else rest.strip()
    if not station_name and bundesland:
        station_name = bundesland
    station_name = re.sub(r"\bFrei\.?$", "", station_name).strip()

    stations_rows.append(parts[:6] + [station_name, bundesland])

stations = pd.DataFrame(
    stations_rows,
    columns=[
        "stations_id", "from_date", "to_date",
        "stationshoehe", "lat", "lon",
        "name", "state"
    ]
)

# Typkonvertierungen
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")
stations["stationshoehe"] = pd.to_numeric(stations["stationshoehe"], errors="coerce")
stations["from_date"] = pd.to_datetime(stations["from_date"], format="%Y%m%d", errors="coerce")
stations["to_date"] = pd.to_datetime(stations["to_date"], format="%Y%m%d", errors="coerce")

stations = stations.dropna(subset=["lat", "lon", "to_date"])
print(f" {len(stations)} historische DWD-Stationen geladen.")

# ------------------------------------------------
# 4️ Finde Stadt- & Landstation (mit Blacklist großer Städte)
# -----------------------------------------------
MIN_RURAL_DISTANCE_KM = 30



# Lade die ersten 300 Städte
urban_exclude_df = pd.read_excel(excel_path, sheet_name="Städte", skiprows=1, nrows=300)

# Nur den ersten Begriff vor dem ersten Leerzeichen nehmen
urban_exclude_df["Stadt_clean"] = (
    urban_exclude_df["Stadt"]
    .astype(str)
    .str.split(",").str[0]     # Entfernt ", Stadt"
    .str.split().str[0]        # Nimmt nur das erste Wort (z. B. "Frankfurt")
    .str.strip()
)

urban_exclude = sorted(urban_exclude_df["Stadt_clean"].dropna().unique().tolist())

print(f" {len(urban_exclude)} Städte auf Blacklist (vereinfachte Variante).")
print(f"Beispiele: {urban_exclude[:15]} ...\n")


# Regex-Muster (case-insensitive)
urban_exclude_pattern = "|".join(map(re.escape, urban_exclude))

print(f" {len(urban_exclude)} große Städte werden für Rural-Stationen ausgeschlossen.")
print(f"Beispiele: {urban_exclude[:10]} ...\n")

def find_nearest_station(lat, lon):
    df = stations[stations["to_date"] >= pd.Timestamp("2024-01-01")].copy()
    df["dist_km"] = df.apply(lambda r: geodesic((lat, lon), (r["lat"], r["lon"])).km, axis=1)
    row = df.loc[df["dist_km"].idxmin()]
    return row["stations_id"], row["name"], row["lat"], row["lon"], row["stationshoehe"], row["dist_km"], row["state"]

def find_rural_station(lat, lon, city_clean):
    """
    Findet die nächste ländliche Station (>=30 km Abstand),
    deren Name weder die untersuchte Stadt noch eine andere große Stadt enthält
    UND die bis mindestens 2024 aktiv ist.
    """
    city_lower = city_clean.lower()

    # Nur Stationen mit gültigen Koordinaten & aktiv bis mind. 2024
    df = stations[
        (stations["to_date"] >= pd.Timestamp("2024-01-01")) &
        (stations["lat"].notna()) &
        (stations["lon"].notna())
    ].copy()

    df["name_lower"] = df["name"].str.lower()

    def is_excluded(name):
        if city_lower in name:
            return True
        for bad in urban_exclude:
            if bad.lower() in name:
                return True
        return False

    df = df[~df["name_lower"].apply(is_excluded)]

    # Distanz berechnen
    df["dist_km"] = df.apply(lambda r: geodesic((lat, lon), (r["lat"], r["lon"])).km, axis=1)

    # Mindestens 30 km Abstand
    df_far = df[df["dist_km"] >= MIN_RURAL_DISTANCE_KM]

    if not df_far.empty:
        row = df_far.loc[df_far["dist_km"].idxmin()]
        valid = True

    if df_far.empty:
        print(f"Keine aktive Rural-Station für {city_clean}")
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, False


    return row["stations_id"], row["name"], row["lat"], row["lon"], row["stationshoehe"], row["dist_km"], valid




# ------------------------------------------------
# Stadt–Land-Zuordnung durchführen
# ------------------------------------------------
nearest, rural = [], []

for _, row in tqdm(cities.iterrows(), total=len(cities)):
    sid, sname, slat, slon, sheight, sdist, sstate = find_nearest_station(row["lat"], row["lon"])
    rid, rname, rlat, rlon, rheight, rdist, valid = find_rural_station(row["lat"], row["lon"], row["city_clean"])
    nearest.append((sid, sname, slat, slon, sheight, sdist, sstate))
    rural.append((rid, rname, rlat, rlon, rheight, rdist, valid))

cities[[
    "station_id", "station_name", "station_lat", "station_lon",
    "station_height", "distance_km", "city_state"
]] = nearest

cities[[
    "rural_station_id", "rural_station_name", "rural_lat", "rural_lon",
    "rural_height", "rural_distance_km", "rural_valid"
]] = rural

print("\n Stadt–Land-Stationen ermittelt:")
print(cities[[
    "city_clean", "city_state", "station_name", "rural_station_name", "rural_distance_km"
]])


# ------------------------------------------------
# 5️ Lade historische Tagesdaten 2019–2024 & berechne Summen
# ------------------------------------------------
print("\n Lade Tagesdaten (historisch, 2019–2024)...")

base_url = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/"

def find_historical_zip_url(station_id):
    sid = f"{int(station_id):05d}"
    html = requests.get(base_url).text
    files = re.findall(rf"tageswerte_KL_{sid}_\d+_\d+_hist\.zip", html)
    if files:
        return base_url + files[-1]
    return None

def count_hot_tropical_days_5yrs(station_id):
    url = find_historical_zip_url(station_id)
    if url is None:
        print(f" Keine historische Datei für Station {station_id}")
        return np.nan, np.nan
    try:
        r = requests.get(url, timeout=25)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            fname = [f for f in z.namelist() if f.startswith("produkt_klima")][0]
            with z.open(fname) as f:
                df = pd.read_csv(f, sep=";", encoding="latin-1", comment="#")
        df.columns = [c.strip().upper() for c in df.columns]
        df["MESS_DATUM"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d", errors="coerce")
        df["TXK"] = pd.to_numeric(df["TXK"], errors="coerce")
        df["TNK"] = pd.to_numeric(df["TNK"], errors="coerce")

        df = df[df["MESS_DATUM"].dt.month.isin([6,7,8])]
        df = df[df["MESS_DATUM"].dt.year.between(2019, 2024)]

        hitzetage = (df["TXK"] >= 30).sum()
        tropennaechte = (df["TNK"] >= 20).sum()

        return hitzetage, tropennaechte

    except Exception as e:
        print(f" Station {station_id}: {e}")
        return np.nan, np.nan

stats = []
for sid, rid in tqdm(zip(cities["station_id"], cities["rural_station_id"]), total=len(cities)):
    h_u, t_u = count_hot_tropical_days_5yrs(sid)
    h_r, t_r = count_hot_tropical_days_5yrs(rid)
    stats.append((h_u, t_u, h_r, t_r))

cities[["Sum_TXK_urban", "Sum_TNK_urban", "Sum_TXK_rural", "Sum_TNK_rural"]] = stats

cities["Diff_TXK"] = cities["Sum_TXK_urban"] - cities["Sum_TXK_rural"]
cities["Diff_TNK"] = cities["Sum_TNK_urban"] - cities["Sum_TNK_rural"]

# ------------------------------------------------
# 6️ Speichern – Excel-kompatible CSV
# ------------------------------------------------
out_path = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\uhi_hitzetage_2019_2024_TEST_V2.csv"

num_cols = [
    "lat", "lon", "station_lat", "station_lon",
    "rural_lat", "rural_lon",
    "distance_km", "rural_distance_km",
    "Sum_TXK_urban", "Sum_TNK_urban",
    "Sum_TXK_rural", "Sum_TNK_rural",
    "Diff_TXK", "Diff_TNK",
]
for col in num_cols:
    if col in cities.columns:
        cities[col] = pd.to_numeric(cities[col], errors="coerce").round(3)

cities.to_csv(
    out_path,
    sep=";", decimal=",", index=False, encoding="utf-8-sig"
)



📥 Lade Städtedaten ...

🏙️ Verwende 300 größte Städte:
                              city  area_km2  population
0                    Berlin, Stadt    891.12   3685265.0
1    Hamburg, Freie und Hansestadt    755.09   1862565.0
2        München, Landeshauptstadt    310.70   1505005.0
3                      Köln, Stadt    405.02   1024621.0
4         Frankfurt am Main, Stadt    248.31    756021.0
..                             ...       ...         ...
295         Langen (Hessen), Stadt     29.12     38785.0
296               Ettlingen, Stadt     56.75     38578.0
297            Niederkassel, Stadt     35.79     38485.0
298                Coesfeld, Stadt    141.36     38237.0
299           Kamp-Lintfort, Stadt     63.14     38217.0

[300 rows x 3 columns]

📍 Ermittle Geokoordinaten ...


100%|██████████| 300/300 [04:58<00:00,  1.00it/s]



📥 Lade DWD-Stationsbeschreibung (historische Tageswerte)...
✅ 1380 historische DWD-Stationen geladen.

🚫 Erstelle vereinfachte Blacklist großer Städte für Rural-Stations ...
➡️ 294 Städte auf Blacklist (vereinfachte Variante).
Beispiele: ['Aachen', 'Aalen', 'Ahaus', 'Ahlen', 'Albstadt', 'Alsdorf', 'Amberg', 'Ansbach', 'Arnsberg', 'Aschaffenburg', 'Augsburg', 'Aurich', 'Backnang', 'Bad', 'Baden-Baden'] ...

➡️ 294 große Städte werden für Rural-Stationen ausgeschlossen.
Beispiele: ['Aachen', 'Aalen', 'Ahaus', 'Ahlen', 'Albstadt', 'Alsdorf', 'Amberg', 'Ansbach', 'Arnsberg', 'Aschaffenburg'] ...



100%|██████████| 300/300 [00:35<00:00,  8.54it/s]



✅ Stadt–Land-Stationen ermittelt:
            city_clean           city_state            station_name  \
0               Berlin               Berlin                  Berlin   
1              Hamburg              Hamburg                 Hamburg   
2              München               Bayern           München-Stadt   
3                 Köln  Nordrhein-Westfalen          Köln-Stammheim   
4    Frankfurt am Main               Hessen  Frankfurt/Main-Westend   
..                 ...                  ...                     ...   
295    Langen (Hessen)               Hessen          Frankfurt/Main   
296          Ettlingen    Baden-Württemberg            Rheinstetten   
297       Niederkassel  Nordrhein-Westfalen               Köln/Bonn   
298           Coesfeld  Nordrhein-Westfalen     Borken in Westfalen   
299      Kamp-Lintfort  Nordrhein-Westfalen          Duisburg-Baerl   

         rural_station_name rural_distance_km  
0                Heckelberg         39.573265  
1               

...

⚠️ Keine historische Datei für Station 06243


 96%|█████████▌| 288/300 [05:10<00:11,  1.03it/s]

⚠️ Keine historische Datei für Station 20098


100%|██████████| 300/300 [05:24<00:00,  1.08s/it]


💾 Ergebnisse gespeichert unter:
➡️ C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\uhi_hitzetage_2019_2024_TEST_V2.csv
✅ Excel-kompatible Formatierung aktiviert (Trennzeichen=';', Dezimal=',').


In [39]:
print("Frankfurt" in "Frankfurt /Main")

True


In [ ]:
import os
import io
import re
import time
from dataclasses import dataclass
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from zipfile import ZipFile, BadZipFile

# -----------------------------------------
#  Basiseinstellungen
# -----------------------------------------
BASE_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/monthly/kl/historical/"
STATION_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/KL_Tageswerte_Beschreibung_Stationen.txt"

ZIP_NAME_RE = re.compile(r"^monatswerte_KL_(\d{5})_(\d{8})_(\d{8})_hist\.zip$")
PRODUKT_RE = re.compile(r"^produkt_klima_monat_.*\.txt$", re.IGNORECASE)
NA_VALUES = [-999, -999.0, "-999", "-999.0"]

SELECTED_COLS = [
    "STATIONS_ID",
    "MESS_DATUM_BEGINN",
    "MESS_DATUM_ENDE",
    "MO_TT",
    "MO_FK",
    "MO_SD_S",
    "MO_RR"
]


@dataclass
class ZipEntry:
    url: str
    station_id: int
    start: str
    end: str
    name: str


# -----------------------------------------
#  Stationen korrekt einlesen (unsere Version)
# -----------------------------------------
def read_dwd_station_description():
    """Liest und bereinigt die DWD-Stationstabelle (historisch, daily)."""
    response = requests.get(STATION_URL)
    response.encoding = "latin1"
    lines = response.text.splitlines()

    bundeslaender = [
    "Baden-Württemberg", "Bayern", "Berlin", "Brandenburg", "Bremen",
    "Hamburg", "Hessen", "Mecklenburg-Vorpommern", "Niedersachsen",
    "Nordrhein-Westfalen", "Rheinland-Pfalz", "Saarland",
    "Sachsen-Anhalt", "Sachsen", "Schleswig-Holstein", "Thüringen"
]


    rows = []
    for line in lines[2:]:  # ab Zeile 3
        if not line.strip():
            continue
        parts = line.split(maxsplit=6)
        if len(parts) < 7:
            continue

        rest = parts[6]
        bundesland = next((bl for bl in bundeslaender if re.search(rf"\b{re.escape(bl)}\b", rest)), None)
        station_name = rest.split(bundesland)[0].strip() if bundesland else rest.strip()
        station_name = re.sub(r"\bFrei\.?$", "", station_name).strip()

        rows.append(parts[:6] + [station_name, bundesland])

    df = pd.DataFrame(
        rows,
        columns=[
            "stations_id", "from_date", "to_date",
            "stationshoehe", "lat", "lon",
            "name", "state"
        ]
    )

    df["stations_id"] = pd.to_numeric(df["stations_id"], errors="coerce").astype("Int64")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    df["stationshoehe"] = pd.to_numeric(df["stationshoehe"], errors="coerce")
    df = df.dropna(subset=["lat", "lon"])
    print(f" {len(df)} DWD-Stationen erfolgreich eingelesen.")
    return df


# -----------------------------------------
#  Hilfsfunktionen (unverändert)
# -----------------------------------------
def sniff_sep(csv_path: str) -> str:
    """Erkennt automatisch das Trennzeichen der CSV."""
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
        head = f.read(2048)
    return ";" if head.count(";") >= head.count(",") else ","


def read_station_ids(csv_path: str):
    """Liest station_id aus der CSV-Datei."""
    sep = sniff_sep(csv_path)
    df = pd.read_csv(csv_path, sep=sep)
    if "station_id" not in df.columns:
        for c in df.columns:
            if c.lower().strip() == "station_id":
                df.rename(columns={c: "station_id"}, inplace=True)
                break
        else:
            raise ValueError("Spalte 'station_id' nicht gefunden.")
    sids = (
        pd.to_numeric(df["station_id"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if not sids:
        raise ValueError("Keine gültigen station_id-Werte gefunden.")
    return sids


def fetch_index(timeout=30, retries=3):
    """Holt die Indexseite der DWD-Historie."""
    for i in range(retries):
        try:
            r = requests.get(BASE_URL, timeout=timeout)
            r.raise_for_status()
            return r.text
        except Exception as e:
            print(f"[WARN] Index-Abruf fehlgeschlagen ({i+1}/{retries}): {e}")
            time.sleep(1)
    raise RuntimeError("Konnte DWD-Index nicht abrufen.")


def parse_index_for_station(index_html: str, station_id: int):
    """Sucht nur ZIP-Dateien mit Enddatum bis 2024."""
    soup = BeautifulSoup(index_html, "html.parser")
    entries = []
    for a in soup.find_all("a", href=True):
        name = a["href"].split("/")[-1]
        m = ZIP_NAME_RE.match(name)
        if not m:
            continue
        sid, start, end = m.groups()
        if int(sid) == station_id and end.startswith("2024"):
            url = urljoin(BASE_URL, name)
            entries.append(ZipEntry(url, station_id, start, end, name))
    return sorted(entries, key=lambda e: e.end)


def download_zip(url: str, timeout=30):
    """Lädt eine ZIP-Datei vom DWD."""
    print(f"[DL] {url}")
    r = requests.get(url, timeout=timeout, stream=True)
    r.raise_for_status()
    return r.content


def extract_klima_from_zip(zip_bytes: bytes, zip_name: str, station_id: int):
    """Extrahiert Daten aus der produkt_klima_monat-Datei."""
    try:
        with ZipFile(io.BytesIO(zip_bytes)) as zf:
            members = [m for m in zf.namelist() if PRODUKT_RE.match(os.path.basename(m))]
            if not members:
                raise FileNotFoundError(f"Keine produkt_klima_monat-Datei in {zip_name}")
            members.sort(key=lambda n: (len(n), n))
            member = members[0]
            print(f"[ZIP] → {member}")
            with zf.open(member) as fh:
                df = pd.read_csv(fh, sep=";", na_values=NA_VALUES, engine="python")
    except BadZipFile as e:
        raise RuntimeError(f"Defektes ZIP {zip_name}: {e}")

    df = df[[c for c in SELECTED_COLS if c in df.columns]].copy()
    if "STATIONS_ID" not in df.columns:
        df["STATIONS_ID"] = station_id
    df["station_id"] = station_id

    for col in ("MESS_DATUM_BEGINN", "MESS_DATUM_ENDE"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col].astype(str), format="%Y%m%d", errors="coerce")

    if "MESS_DATUM_BEGINN" in df.columns:
        mask = (
            (df["MESS_DATUM_BEGINN"].dt.year >= 2019)
            & (df["MESS_DATUM_BEGINN"].dt.year <= 2024)
            & (df["MESS_DATUM_BEGINN"].dt.month.isin([6, 7, 8]))
        )
        df = df.loc[mask].copy()

    return df


# -----------------------------------------
#  Hauptfunktion für Notebook-Nutzung
# -----------------------------------------
def download_dwd_klima_sommer_mittelwerte(input_csv, output_dir, filename="dwd_klima_sommer_2019_2024_mittelwerte.xlsx"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, filename)

    stations = read_dwd_station_description()
    station_ids = read_station_ids(input_csv)

    print(f"[INFO] {len(station_ids)} Stationen aus CSV geladen.")
    index_html = fetch_index()

    all_data = []
    for sid in station_ids:
        entries = parse_index_for_station(index_html, sid)
        if not entries:
            print(f"[WARN] Keine ZIP-Dateien bis 2024 für Station {sid} gefunden.")
            continue

        for e in entries:
            try:
                zip_bytes = download_zip(e.url)
                df = extract_klima_from_zip(zip_bytes, e.name, e.station_id)
                if not df.empty:
                    df["source_zip"] = e.name
                    all_data.append(df)
                time.sleep(0.5)
            except Exception as ex:
                print(f"[ERROR] {e.name}: {ex}")

    if not all_data:
        raise RuntimeError("Keine Daten extrahiert!")

    result = pd.concat(all_data, ignore_index=True)

    mean_df = (
        result.groupby("STATIONS_ID")[["MO_TT", "MO_FK", "MO_SD_S", "MO_RR"]]
        .mean()
        .reset_index()
        .rename(columns={
            "MO_TT": "mean_MO_TT",
            "MO_FK": "mean_MO_FK",
            "MO_SD_S": "mean_MO_SD_S",
            "MO_RR": "mean_MO_RR",
        })
    )

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        result.to_excel(writer, index=False, sheet_name="DWD_KL_Sommer_2019_2024")
        mean_df.to_excel(writer, index=False, sheet_name="Mittelwerte_pro_Station")

    print(f"[OK] Rohdaten: {len(result)} Zeilen, Mittelwerte: {len(mean_df)} Stationen gespeichert unter:\n  {output_path}")
    return result, mean_df


In [50]:
# Pfade anpassen:
input_csv = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\uhi_hitzetage_2019_2024.csv"
output_dir = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt"


raw_df, mean_df = download_dwd_klima_sommer_mittelwerte(
    input_csv=input_csv,
    output_dir=output_dir,
    filename="dwd_klima_sommer_2019_2024_mittelwerte.xlsx"
)


✅ 1380 DWD-Stationen erfolgreich eingelesen.
[INFO] 181 Stationen aus CSV geladen.
[DL] https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/monthly/kl/historical/monatswerte_KL_00433_19380101_20241231_hist.zip
[ZIP] → produkt_klima_monat_19380101_20241231_00433.txt
[DL] https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/monthly/kl/historical/monatswerte_KL_01975_19360101_20241231_hist.zip
[ZIP] → produkt_klima_monat_19360101_20241231_01975.txt
[DL] https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/monthly/kl/historical/monatswerte_KL_03379_19540601_20241231_hist.zip
[ZIP] → produkt_klima_monat_19540601_20241231_03379.txt
[DL] https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/monthly/kl/historical/monatswerte_KL_02968_19030101_20240831_hist.zip
[ZIP] → produkt_klima_monat_19030101_20240831_02968.txt
[DL] https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/mon

In [ ]:
import pandas as pd
import os

def merge_uhi_and_dwd(
    uhi_file: str,
    dwd_excel: str,
    output_dir: str,
    output_filename: str = "uhi_dwd_merged.xlsx",
):
    """
    Verbindet eine UHI-Tabelle (CSV oder Excel) mit den DWD-Mittelwerten über 'station_id'.
    """

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_filename)

    # --- 1️ UHI-Daten laden ---
    if uhi_file.lower().endswith(".csv"):
        try:
            uhi_df = pd.read_csv(uhi_file, sep=";", decimal=",")
        except Exception:
            uhi_df = pd.read_csv(uhi_file)
    elif uhi_file.lower().endswith((".xls", ".xlsx")):
        uhi_df = pd.read_excel(uhi_file)
    else:
        raise ValueError("Unbekanntes Dateiformat für UHI-Datei (nur .csv oder .xlsx unterstützt)")

    # station_id angleichen
    for c in uhi_df.columns:
        if c.lower().strip() == "station_id":
            uhi_df.rename(columns={c: "station_id"}, inplace=True)
    if "station_id" not in uhi_df.columns:
        raise ValueError("UHI-Datei enthält keine Spalte 'station_id'.")

    print(f"[INFO] UHI-Tabelle geladen: {uhi_df.shape[0]} Zeilen")

    # --- 2️ DWD-Daten laden ---
    xls = pd.ExcelFile(dwd_excel)
    if "Mittelwerte_pro_Station" not in xls.sheet_names:
        raise ValueError("Das Blatt 'Mittelwerte_pro_Station' wurde nicht gefunden.")
    dwd_df = pd.read_excel(xls, sheet_name="Mittelwerte_pro_Station")

    if "STATIONS_ID" in dwd_df.columns:
        dwd_df.rename(columns={"STATIONS_ID": "station_id"}, inplace=True)

    print(f"[INFO] DWD-Tabelle geladen: {dwd_df.shape[0]} Stationen")

    # --- 3️ Zusammenführen ---
    merged = pd.merge(uhi_df, dwd_df, on="station_id", how="left")
    print(f"[OK] Verbundene Tabelle: {merged.shape[0]} Zeilen, {merged.shape[1]} Spalten")

    # --- 4️ Ausgabe speichern ---
    merged.to_excel(output_path, index=False, sheet_name="UHI_DWD_Merged")
    print(f"[SAVED] Ergebnis gespeichert unter: {output_path}")

    return merged


In [58]:
uhi_file = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\uhi_hitzetage_2019_2024_TEST_V2.csv"
dwd_excel = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\dwd_klima_sommer_2019_2024_mittelwerte.xlsx"
output_dir = r"C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt"

merged_df = merge_uhi_and_dwd(
    uhi_file=uhi_file,
    dwd_excel=dwd_excel,
    output_dir=output_dir,
    output_filename="uhi_dwd_merged_V2.xlsx"
)


[INFO] UHI-Tabelle geladen: 300 Zeilen
[INFO] DWD-Tabelle geladen: 165 Stationen
[OK] Verbundene Tabelle: 300 Zeilen, 30 Spalten
[SAVED] Ergebnis gespeichert unter: C:\Users\jetmi\Unterlagen_Jetmir\Data_Science\Abschlussprojekt\uhi_dwd_merged_V2.xlsx
